# 00 - Setup: your first call to Claude

**ICSSI 2026, Using LLMs via the API**

The goal for the next five minutes is to make a real API call and understand what comes
back. By the end you will have the SDK installed, your key loaded, and a working call
that summarizes a paper abstract, along with the token usage that the call cost.

### Before you run anything

If you are in Google Colab there is nothing to do, because the first code cell installs
the `anthropic` package for you. Just add your key in the Secrets panel (the key icon) as
`ANTHROPIC_API_KEY` and run the cells.

If you are running locally you need a Python environment with Jupyter already running
before any cell can execute. The bootstrap cell can only install the `anthropic` package,
not the notebook runtime itself. Do this one-time setup first, from the repo folder:

```bash
python3 -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\activate
pip install -r requirements.txt    # installs anthropic and JupyterLab
cp .env.example .env               # then paste your key into .env
jupyter lab                        # then open this notebook and pick the .venv kernel
```

If you prefer VS Code or Cursor, open the folder and select the `.venv` kernel instead of
launching JupyterLab. Full details are in the README. You do not need Docker or conda, and
Python 3.10 or newer is required. This notebook needs no other repo files.

## Step 1: install the SDK and load your API key

Run the cell below. It installs the `anthropic` package if needed and finds your key from
a Colab secret or a local `.env` file. If it complains, see the README for how to set
`ANTHROPIC_API_KEY`.

In [ ]:
# bootstrap: works in Google Colab and locally
import os, sys, subprocess

def pip_install(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

try:
    import anthropic
except ImportError:
    pip_install("anthropic")
    import anthropic

# find the api key: env var first, then a Colab secret, then a local .env file
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    try:
        from dotenv import load_dotenv
    except ImportError:
        pip_install("python-dotenv")
        from dotenv import load_dotenv
    load_dotenv(override=True)

assert os.environ.get("ANTHROPIC_API_KEY"), (
    "No ANTHROPIC_API_KEY found. Running locally, copy .env.example to .env and add your "
    "key. In Colab, add it in the Secrets panel (the key icon), named ANTHROPIC_API_KEY."
)
print("anthropic version:", anthropic.__version__, "| API key loaded: OK")
MODEL = "claude-sonnet-4-6"

## Step 2: one call to the Messages API

Everything goes through `client.messages.create(...)`. You pick a model, a cap on the
reply length called `max_tokens`, and a list of messages. The API is stateless, so it
only knows what you send in this one request.

In [ ]:
tclient = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment

# Teplitskiy et al, Research Policy, 2022
abstract = (
    """
    Although citations are widely used to measure the influence of scientific works, research shows that many citations serve rhetorical functions and reflect little-to-no influence on the citing authors. If highly cited papers disproportionately attract rhetorical citations then their citation counts may reflect rhetorical usefulness more than influence. Alternatively, researchers may perceive highly cited papers to be of higher quality and invest more effort into reading them, leading to disproportionately substantive citations. We test these arguments using data on 17,154 randomly sampled citations collected via surveys from 9,380 corresponding authors in 15 fields. We find that most citations (54%) had little-to-no influence on the citing authors. However, citations to the most highly cited papers were 2–3 times more likely to denote substantial influence. Experimental and correlational data show a key mechanism: displaying low citation counts lowers perceptions of a paper's quality, and papers with poor perceived quality are read more superficially. The results suggest that higher citation counts lead to more meaningful engagement from readers and, consequently, the most highly cited papers influence the research frontier much more than their raw citation counts imply.
    """
)

print(f"Submitting abstract summarization request to {MODEL}...")
response = client.messages.create(
    model=MODEL,
    max_tokens=200,
    messages=[
        {"role": "user", "content": "Summarize this abstract in one sentence:\n\n" + abstract},
    ],
)

# the reply is a list of content blocks, so read the text block
print(f"Response from {MODEL}:\n{response.content[0].text}")

## Step 3: what did that cost? Read the `usage`

Every response carries a usage object. The two numbers that matter most are the input
tokens and the output tokens. The input tokens count everything you sent, which is your
prompt, and you pay for those. The output tokens count what the model generated, and they
cost more per token than input tokens. For Opus 4.8 the rate is five dollars per million
input tokens and twenty-five dollars per million output tokens.

In [ ]:
usage = response.usage
print("input tokens :", usage.input_tokens)
print("output tokens:", usage.output_tokens)

cost = usage.input_tokens * 5.0 / 1_000_000 + usage.output_tokens * 25.0 / 1_000_000
print(f"this call cost: ${cost:.6f}")

## Recap and what is next

There is one endpoint, `client.messages.create(model, max_tokens, messages)`. The reply
is in `response.content` and the bill is in `response.usage`. Output tokens are the
expensive ones, so keep replies as short as the task allows.

A note on **key safety**: never paste your API key (`sk-ant-...`) into a code cell, capture it in a screenshot, or version it in a git repository. Use a `.env` file
locally or the Colab Secrets panel, both of which are git-ignored.

Next up is `01_cost.ipynb`, which shows how to measure and minimize what you spend on a real
abstract-classification task.